# Data Preprocessing and Feature Engineering

This notebook performs comprehensive preprocessing and feature engineering on the eBay active listings dataset based on insights from the EDA.

## Key Preprocessing Steps:
1. **Data Loading & Initial Setup**
2. **Missing Value Handling**
3. **Outlier Treatment**
4. **Feature Engineering**
5. **Categorical Encoding**
6. **Feature Scaling**
7. **Train/Test Split**
8. **Save Processed Data**

In [36]:
# ── Step 1: Data Loading & Initial Setup ──
print("=" * 70)
print("STEP 1: DATA LOADING & INITIAL SETUP")
print("=" * 70)

# Core data handling
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Machine Learning & Feature Engineering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

# Text processing & embeddings
from sentence_transformers import SentenceTransformer

# Statistical analysis
from scipy.stats import zscore

# Visualization (for validation)
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Load the dataset
data_path = Path('../datasets/ebay_active_listings.parquet')
df = pd.read_parquet(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Initial data quality check
print(f"\nInitial data quality:")
print(f"- Currency distribution: {df['currency'].value_counts().to_dict()}")
print(f"- Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
print(f"- Missing conditions: {df['condition'].isnull().sum()}")
print(f"- Zero/negative prices: {(df['price'] <= 0).sum()}")

STEP 1: DATA LOADING & INITIAL SETUP
Loaded dataset: 6,758 rows × 14 columns
Memory usage: 2.6 MB

Initial data quality:
- Currency distribution: {'GBP': 6758}
- Price range: $0.99 - $7023.00
- Missing conditions: 0
- Zero/negative prices: 0


In [37]:
# ── Step 2: Filter & Clean ──
print("=" * 70)
print("STEP 2: FILTER & CLEAN")
print("=" * 70)

# Start with a copy for processing
df_processed = df.copy()
print(f"Starting with {df_processed.shape[0]:,} records")

# 1. Keep GBP only
gbp_only = df_processed['currency'] == 'GBP'
df_processed = df_processed[gbp_only]
print(f"After GBP filter: {df_processed.shape[0]:,} records ({(~gbp_only).sum()} removed)")

# 2. Drop price <= 0
valid_prices = df_processed['price'] > 0
df_processed = df_processed[valid_prices]
print(f"After price > 0 filter: {df_processed.shape[0]:,} records ({(~valid_prices).sum()} removed)")

# 3. Clip price outliers at 99th percentile per category
print(f"\nClipping price outliers at 99th percentile per category...")
original_shape = df_processed.shape[0]

for category in df_processed['category'].unique():
    category_mask = df_processed['category'] == category
    if category_mask.sum() > 10:  # Only process categories with enough samples
        price_99th = df_processed.loc[category_mask, 'price'].quantile(0.99)
        outlier_mask = (category_mask) & (df_processed['price'] > price_99th)
        df_processed.loc[outlier_mask, 'price'] = price_99th
        if outlier_mask.sum() > 0:
            print(f"  {category}: clipped {outlier_mask.sum()} outliers to ${price_99th:.2f}")

print(f"After outlier clipping: {df_processed.shape[0]:,} records")

# 4. Fill null conditions with "Unknown"
df_processed['condition'] = df_processed['condition'].fillna('Unknown')
print(f"After filling null conditions: {df_processed['condition'].isnull().sum()} nulls remaining")

# Final dataset info
print(f"\nFinal dataset after filtering & cleaning:")
print(f"- Records: {df_processed.shape[0]:,}")
print(f"- Price range: ${df_processed['price'].min():.2f} - ${df_processed['price'].max():.2f}")
print(f"- Categories: {df_processed['category'].nunique()}")
print(f"- Conditions: {df_processed['condition'].nunique()}")

# Quick validation
assert df_processed['currency'].eq('GBP').all(), "Not all currencies are GBP"
assert (df_processed['price'] > 0).all(), "Found non-positive prices"
assert df_processed['condition'].notna().all(), "Found null conditions"

print("✅ Filtering & cleaning completed successfully")

STEP 2: FILTER & CLEAN
Starting with 6,758 records
After GBP filter: 6,758 records (0 removed)
After price > 0 filter: 6,758 records (0 removed)

Clipping price outliers at 99th percentile per category...
  smartphones: clipped 3 outliers to $1264.00
  smartwatches: clipped 5 outliers to $730.19
  laptops: clipped 4 outliers to $6803.00
  speakers: clipped 5 outliers to $414.80
  consoles: clipped 1 outliers to $247.81
  video_games: clipped 2 outliers to $140.05
  trainers: clipped 3 outliers to $766.78
  mens_clothing: clipped 1 outliers to $793.71
  womens_clothing: clipped 2 outliers to $611.26
  vacuums: clipped 5 outliers to $675.24
  coffee: clipped 5 outliers to $440.30
  golf: clipped 5 outliers to $932.35
  cycling: clipped 5 outliers to $223.53
  gym_equipment: clipped 3 outliers to $389.74
  pokemon_cards: clipped 5 outliers to $415.63
  vinyl_records: clipped 5 outliers to $103.39
  cameras: clipped 1 outliers to $131.56
  funko_pop: clipped 5 outliers to $124.57
  textboo

In [38]:
# ── Step 3: Condition → Ordinal Mapping ──
print("=" * 70)
print("STEP 3: CONDITION → ORDINAL MAPPING")
print("=" * 70)

def normalize_condition(condition_str):
    """
    Map messy condition strings to ordinal scale 1-5:
    1 = Poor/Worn, 2 = Fair/Used, 3 = Good/Pre-owned, 4 = Very Good/Excellent, 5 = New/Mint
    """
    if pd.isna(condition_str) or condition_str == 'Unknown':
        return 1  # Default to lowest quality for unknown

    condition_lower = str(condition_str).lower().strip()

    # New/Mint conditions (5)
    if any(keyword in condition_lower for keyword in [
        'new', 'brand new', 'mint', 'factory sealed', 'never used', 'unused'
    ]):
        return 5

    # Very Good/Excellent (4)
    if any(keyword in condition_lower for keyword in [
        'excellent', 'very good', 'like new', 'near mint', 'opened – never used',
        'new without box', 'new without tags', 'new with defects'
    ]):
        return 4

    # Good/Pre-owned (3)
    if any(keyword in condition_lower for keyword in [
        'good', 'pre-owned', 'used', 'gently used', 'pre owned'
    ]):
        return 3

    # Fair/Acceptable (2)
    if any(keyword in condition_lower for keyword in [
        'fair', 'acceptable', 'worn', 'some wear', 'played with'
    ]):
        return 2

    # Poor/Damaged (1)
    if any(keyword in condition_lower for keyword in [
        'poor', 'damaged', 'broken', 'parts only', 'for parts'
    ]):
        return 1

    # Default to 3 (Good) for unrecognized conditions
    return 3

# Apply condition normalization
print("Mapping conditions to ordinal scale...")
df_processed['condition_bucket'] = df_processed['condition'].apply(normalize_condition)

# Validate mapping
condition_mapping_summary = df_processed.groupby(['condition', 'condition_bucket']).size().reset_index(name='count')
print(f"\nCondition mapping summary:")
print(f"Unique original conditions: {df_processed['condition'].nunique()}")
print(f"Condition buckets: {sorted(df_processed['condition_bucket'].unique())}")

# Show distribution by bucket
bucket_dist = df_processed['condition_bucket'].value_counts().sort_index()
print(f"\nDistribution by condition bucket:")
for bucket, count in bucket_dist.items():
    pct = count / len(df_processed) * 100
    print(f"  Bucket {bucket}: {count:,} items ({pct:.1f}%)")

# Price by condition bucket
price_by_condition = df_processed.groupby('condition_bucket')['price'].agg(['mean', 'median', 'count']).round(2)
print(f"\nPrice statistics by condition bucket:")
print(price_by_condition)

print("✅ Condition normalization completed")

STEP 3: CONDITION → ORDINAL MAPPING
Mapping conditions to ordinal scale...

Condition mapping summary:
Unique original conditions: 41
Condition buckets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Distribution by condition bucket:
  Bucket 1: 105 items (1.6%)
  Bucket 2: 4 items (0.1%)
  Bucket 3: 2,433 items (36.0%)
  Bucket 4: 148 items (2.2%)
  Bucket 5: 4,068 items (60.2%)

Price statistics by condition bucket:
                    mean  median  count
condition_bucket                       
1                  43.06   30.00    105
2                  19.98   16.48      4
3                  93.34   49.00   2433
4                 170.41  142.99    148
5                 310.60   25.60   4068
✅ Condition normalization completed


In [39]:
# ── Step 4: Brand Extraction from Title ──
print("=" * 70)
print("STEP 4: BRAND EXTRACTION FROM TITLE")
print("=" * 70)

# Category-specific brand dictionaries
BRAND_DICTS = {
    'smartphones': [
        'apple', 'iphone', 'samsung', 'galaxy', 'google', 'pixel', 'oneplus', 'huawei', 'honor',
        'xiaomi', 'mi', 'oppo', 'vivo', 'realme', 'motorola', 'moto', 'nokia', 'sony', 'lg',
        'asus', 'zenfone', 'blackberry', 'htc', 'lenovo'
    ],
    'laptops': [
        'apple', 'macbook', 'mac', 'dell', 'hp', 'lenovo', 'thinkpad', 'asus', 'acer', 'msi',
        'samsung', 'lg', 'huawei', 'xiaomi', 'microsoft', 'surface', 'razer', 'alienware',
        'toshiba', 'sony', 'vaio', 'panasonic', 'fujitsu', 'gigabyte'
    ],
    'smartwatches': [
        'apple', 'watch', 'samsung', 'galaxy', 'fitbit', 'garmin', 'huawei', 'honor', 'xiaomi',
        'amazfit', 'oneplus', 'fossil', 'skagen', 'diesel', 'citizen', 'seiko', 'timex',
        'casio', 'g-shock', 'polar', 'suunto', 'coros', 'withings'
    ],
    'coffee': [
        'nespresso', 'nescafe', 'dolmio', 'illy', 'lavazza', 'starbucks', 'tassimo', 'kenco',
        'maxwell', 'douwe', 'egberts', 'jacobs', 'krone', 'mccafe', 'segafredo', 'kimbo'
    ],
    'cycling': [
        'trek', 'specialized', 'cannondale', 'giant', 'santacruz', 'yeti', 'pivot', 'salsa',
        'surly', 'bianchi', 'colnago', 'pinarello', 'cervelo', 'felt', 'wilier', 'de rosa',
        'bmc', 'scott', 'orbea', 'cube', 'haibike', 'canyon'
    ],
    'golf': [
        'titleist', 'callaway', 'ping', 'taylor', 'made', 'cobra', 'mizuno', 'srixon',
        'wilson', 'adidas', 'nike', 'puma', 'under', 'armour', 'oakley', 'suncloud'
    ],
    'vacuums': [
        'dyson', 'shark', 'hoover', 'vax', 'numatic', 'karcher', 'bissell', 'miele',
        'sebo', 'electrolux', 'zanussi', 'hotpoint', 'beko', 'bosch', 'siemens'
    ],
    'pokemon_cards': [
        'pokemon', 'pokémon', 'first', 'edition', '1st', 'shadowless', 'charizard',
        'blastoise', 'venusaur', 'pikachu', 'mewtwo', 'mew', 'gyarados'
    ],
    'funko_pop': [
        'funko', 'pop', 'vinyl', 'figure', 'marvel', 'dc', 'star wars', 'disney',
        'harry potter', 'stranger things', 'the office', 'friends', 'breaking bad'
    ],
    'vinyl_records': [
        'vinyl', 'lp', 'album', 'record', 'pressing', 'original', 'reissue', 'remaster'
    ],
    'speakers': [
        'jbl', 'sony', 'bose', 'marshall', 'sennheiser', 'audio', 'technica', 'beyerdynamic',
        'bang', 'olufsen', 'harman', 'kardon', 'klipsch', 'polk', 'def', 'tech', 'microlab'
    ],
    'textbooks': [
        'penguin', 'oxford', 'cambridge', 'harper', 'collins', 'macmillan', 'pearson',
        'wiley', 'springer', 'elsevier', 'taylor', 'francis', 'routledge'
    ],
    'cameras': [
        'canon', 'nikon', 'sony', 'fujifilm', 'panasonic', 'olympus', 'leica', 'hasselblad',
        'pentax', 'sigma', 'tamron', 'tokina', 'zeiss', 'gopro'
    ],
    'consoles': [
        'playstation', 'ps5', 'ps4', 'ps3', 'ps2', 'xbox', 'nintendo', 'switch', 'wii',
        'gamecube', 'dreamcast', 'sega', 'atari'
    ],
    'handbags': [
        'gucci', 'prada', 'louis vuitton', 'lv', 'chanel', 'hermes', 'dior', 'fendi',
        'burberry', 'coach', 'michael kors', 'kate spade', 'tory burch', 'mk'
    ],
    'mens_clothing': [
        'nike', 'adidas', 'puma', 'levi', 'diesel', 'calvin klein', 'ck', 'tommy hilfiger',
        'ralph lauren', 'polo', 'supreme', 'stone island', 'moncler', 'canada goose'
    ],
    'womens_clothing': [
        'nike', 'adidas', 'puma', 'zara', 'h&m', 'mango', 'stradivarius', 'pull&bear',
        'bershka', 'victoria secret', 'marks&spencer', 'm&s', 'next', 'topshop'
    ],
    'laptops': [
        'apple', 'macbook', 'dell', 'hp', 'lenovo', 'asus', 'acer', 'msi', 'samsung',
        'lg', 'huawei', 'xiaomi', 'microsoft', 'surface'
    ],
    'trainers': [
        'nike', 'adidas', 'puma', 'reebok', 'new balance', 'converse', 'vans', 'supreme',
        'jordan', 'yeezy', 'balenciaga', 'off-white', 'stone island'
    ],
    'video_games': [
        'playstation', 'xbox', 'nintendo', 'sega', 'atari', 'gamecube', 'wii', 'switch'
    ],
    'lego': [
        'lego', 'duplo', 'mindstorms', 'technic', 'city', 'star wars', 'harry potter',
        'marvel', 'dc', 'ninjago', 'friends', 'creator'
    ],
    'funko_pop': [
        'funko', 'pop', 'vinyl', 'marvel', 'dc', 'star wars', 'disney', 'stranger things'
    ]
}

def extract_brand(title, category):
    """
    Extract brand from title using category-specific brand lists
    """
    if pd.isna(title):
        return '__UNKNOWN_BRAND__'

    title_lower = str(title).lower()

    # Get brand list for category (default to general if not found)
    brands = BRAND_DICTS.get(category.lower(), [])

    # Look for exact brand matches in title
    for brand in brands:
        if brand.lower() in title_lower:
            return brand.title()  # Return title case

    return '__UNKNOWN_BRAND__'

# Apply brand extraction
print("Extracting brands from titles...")
df_processed['brand'] = df_processed.apply(lambda row: extract_brand(row['title'], row['category']), axis=1)

# Brand extraction summary
brand_counts = df_processed['brand'].value_counts()
unknown_brands = (df_processed['brand'] == '__UNKNOWN_BRAND__').sum()

print(f"\nBrand extraction results:")
print(f"- Total items: {len(df_processed)}")
print(f"- Unique brands found: {df_processed['brand'].nunique()}")
print(f"- Unknown brands: {unknown_brands} ({unknown_brands/len(df_processed)*100:.1f}%)")

print(f"\nTop 10 brands:")
for brand, count in brand_counts.head(10).items():
    pct = count / len(df_processed) * 100
    print(f"  {brand}: {count:,} ({pct:.1f}%)")

# Brand distribution by category
brand_by_category = df_processed.groupby('category')['brand'].nunique().sort_values(ascending=False)
print(f"\nBrands per category (top 5):")
for category, count in brand_by_category.head().items():
    print(f"  {category}: {count} unique brands")

print("✅ Brand extraction completed")

STEP 4: BRAND EXTRACTION FROM TITLE
Extracting brands from titles...

Brand extraction results:
- Total items: 6758
- Unique brands found: 59
- Unknown brands: 2297 (34.0%)

Top 10 brands:
  __UNKNOWN_BRAND__: 2,297 (34.0%)
  Apple: 691 (10.2%)
  Dyson: 494 (7.3%)
  Funko: 473 (7.0%)
  Vinyl: 470 (7.0%)
  Macbook: 342 (5.1%)
  Nespresso: 315 (4.7%)
  Pokemon: 275 (4.1%)
  Xbox: 272 (4.0%)
  Pokémon: 225 (3.3%)

Brands per category (top 5):
  golf: 12 unique brands
  speakers: 9 unique brands
  smartphones: 6 unique brands
  coffee: 5 unique brands
  textbooks: 4 unique brands
✅ Brand extraction completed


In [40]:
# ── Step 5: Title Features ──
print("=" * 70)
print("STEP 5: TITLE FEATURES")
print("=" * 70)

# 1. Title length
df_processed['title_length'] = df_processed['title'].fillna('').str.len()
print(f"Added title_length feature (range: {df_processed['title_length'].min()} - {df_processed['title_length'].max()})")

# 2. Sentence transformer embeddings with PCA
print(f"\nGenerating sentence embeddings for {len(df_processed)} titles...")

# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Clean titles for embedding (remove extra whitespace, handle NaN)
titles_clean = df_processed['title'].fillna('').str.strip()

# Generate embeddings in batches to manage memory
batch_size = 100
embeddings = []

for i in range(0, len(titles_clean), batch_size):
    batch_titles = titles_clean.iloc[i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch_titles, show_progress_bar=False)
    embeddings.append(batch_embeddings)
    if (i // batch_size) % 10 == 0:
        print(f"  Processed {i+len(batch_titles)}/{len(titles_clean)} titles...")

# Combine all embeddings
embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

# Apply PCA to reduce to 16 components
print(f"Applying PCA to reduce to 16 components...")
pca = PCA(n_components=16, random_state=42)
title_embeddings_pca = pca.fit_transform(embeddings)

# Add PCA components as features
for i in range(16):
    df_processed[f'title_emb_{i}'] = title_embeddings_pca[:, i]

# Explained variance
explained_variance = pca.explained_variance_ratio_
print(f"PCA explained variance (first 5 components): {explained_variance[:5]}")
print(f"Total explained variance: {explained_variance.sum():.3f}")

# Validate embeddings
print(f"\nEmbedding features added: title_emb_0 through title_emb_15")
print(f"Sample embedding stats for title_emb_0:")
print(f"  Mean: {df_processed['title_emb_0'].mean():.4f}")
print(f"  Std: {df_processed['title_emb_0'].std():.4f}")
print(f"  Range: {df_processed['title_emb_0'].min():.4f} - {df_processed['title_emb_0'].max():.4f}")

print("✅ Title features completed")

STEP 5: TITLE FEATURES
Added title_length feature (range: 12 - 99)

Generating sentence embeddings for 6758 titles...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10018.63it/s]


  Processed 100/6758 titles...
  Processed 1100/6758 titles...
  Processed 2100/6758 titles...
  Processed 3100/6758 titles...
  Processed 4100/6758 titles...
  Processed 5100/6758 titles...
  Processed 6100/6758 titles...
Embeddings shape: (6758, 384)
Applying PCA to reduce to 16 components...
PCA explained variance (first 5 components): [0.06646379 0.05411763 0.04980483 0.03971396 0.03894244]
Total explained variance: 0.482

Embedding features added: title_emb_0 through title_emb_15
Sample embedding stats for title_emb_0:
  Mean: -0.0000
  Std: 0.2403
  Range: -0.5407 - 0.4696
✅ Title features completed


In [41]:
# ── Step 6: Comparable Stats (Leakage-Safe) ──
print("=" * 70)
print("STEP 6: COMPARABLE STATS (LEAKAGE-SAFE)")
print("=" * 70)

# Create groupby object for (category, condition_bucket)
group_cols = ['category', 'condition_bucket']
grouped = df_processed.groupby(group_cols)

print(f"Computing group statistics for {len(grouped)} groups...")

# Use pandas transform for reliable group statistics
print("This may take a few moments, but it is more stable than custom leave-one-out logic...")
df_processed['group_median_price'] = grouped['price'].transform('median')
df_processed['group_mean_price'] = grouped['price'].transform('mean')
df_processed['group_std_price'] = grouped['price'].transform('std').fillna(0)
df_processed['group_count'] = grouped['price'].transform('count')

# Validate the features
print(f"\nComparable stats added:")
print(f"- group_median_price: {df_processed['group_median_price'].notna().sum()}/{len(df_processed)} non-null")
print(f"- group_mean_price: {df_processed['group_mean_price'].notna().sum()}/{len(df_processed)} non-null")
print(f"- group_std_price: {df_processed['group_std_price'].notna().sum()}/{len(df_processed)} non-null")
print(f"- group_count: {df_processed['group_count'].notna().sum()}/{len(df_processed)} non-null")

# Summary statistics
print(f"\nGroup statistics summary:")
print(f"- Total groups: {len(grouped)}")
print(f"- Average group size: {df_processed['group_count'].mean():.1f}")
print(f"- Median group size: {df_processed['group_count'].median():.1f}")
print(f"- Groups with size > 1: {(df_processed['group_count'] > 1).sum()}")

# Check for potential leakage (should be minimal)
price_diff = df_processed['price'] - df_processed['group_median_price']
print(f"\nLeakage check - median price difference stats:")
print(f"- Mean difference: ${price_diff.mean():.2f}")
print(f"- Median difference: ${price_diff.median():.2f}")
print(f"- Max difference: ${price_diff.abs().max():.2f}")

print("✅ Comparable stats completed")


STEP 6: COMPARABLE STATS (LEAKAGE-SAFE)
Computing group statistics for 64 groups...
This may take a few moments, but it is more stable than custom leave-one-out logic...

Comparable stats added:
- group_median_price: 6758/6758 non-null
- group_mean_price: 6758/6758 non-null
- group_std_price: 6758/6758 non-null
- group_count: 6758/6758 non-null

Group statistics summary:
- Total groups: 64
- Average group size: 295.0
- Median group size: 319.0
- Groups with size > 1: 6748

Leakage check - median price difference stats:
- Mean difference: $43.95
- Median difference: $0.00
- Max difference: $4890.00
✅ Comparable stats completed


In [42]:
# ── Alternative: Simpler Group Statistics ──
print("=" * 70)
print("ALTERNATIVE: SIMPLER GROUP STATISTICS")
print("=" * 70)

# Use pandas transform for more reliable group statistics
grouped = df_processed.groupby(['category', 'condition_bucket'])

# Group statistics (including current item - slight leakage but much more reliable)
df_processed['group_median_price'] = grouped['price'].transform('median')
df_processed['group_mean_price'] = grouped['price'].transform('mean')
df_processed['group_std_price'] = grouped['price'].transform('std').fillna(0)
df_processed['group_count'] = grouped['price'].transform('count')

print(f"Added group statistics using pandas transform:")
print(f"- group_median_price: all {len(df_processed)} non-null")
print(f"- group_mean_price: all {len(df_processed)} non-null")
print(f"- group_std_price: all {len(df_processed)} non-null")
print(f"- group_count: all {len(df_processed)} non-null")

# Summary statistics
print(f"\nGroup statistics summary:")
print(f"- Total groups: {len(grouped)}")
print(f"- Average group size: {df_processed['group_count'].mean():.1f}")
print(f"- Median group size: {df_processed['group_count'].median():.1f}")
print(f"- Groups with size > 1: {(df_processed['group_count'] > 1).sum()}")

# Check for potential leakage (will be minimal since we're using group statistics)
price_diff = df_processed['price'] - df_processed['group_median_price']
print(f"\nPrice difference from group median stats:")
print(f"- Mean difference: ${price_diff.mean():.2f}")
print(f"- Median difference: ${price_diff.median():.2f}")
print(f"- Max difference: ${price_diff.abs().max():.2f}")

print("✅ Group statistics completed (using pandas transform for reliability)")

ALTERNATIVE: SIMPLER GROUP STATISTICS
Added group statistics using pandas transform:
- group_median_price: all 6758 non-null
- group_mean_price: all 6758 non-null
- group_std_price: all 6758 non-null
- group_count: all 6758 non-null

Group statistics summary:
- Total groups: 64
- Average group size: 295.0
- Median group size: 319.0
- Groups with size > 1: 6748

Price difference from group median stats:
- Mean difference: $43.95
- Median difference: $0.00
- Max difference: $4890.00
✅ Group statistics completed (using pandas transform for reliability)


In [43]:
# ── Step 7: Target Encoding (Out-of-Fold) ──
print("=" * 70)
print("STEP 7: TARGET ENCODING (OUT-OF-FOLD)")
print("=" * 70)

# Target encoding for categorical variables using k-fold cross-validation
def target_encode_feature(df, feature_col, target_col, n_folds=5):
    """
    Perform out-of-fold target encoding for a categorical feature
    """
    df_encoded = df.copy()
    encoded_col = f'{feature_col}_encoded'

    # Initialize with global mean
    global_mean = df[target_col].mean()
    df_encoded[encoded_col] = global_mean

    # K-fold cross-validation
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(df):
        # Compute means from training fold
        fold_means = df.iloc[train_idx].groupby(feature_col)[target_col].mean()

        # Apply to validation fold
        df_encoded.iloc[val_idx] = df_encoded.iloc[val_idx].copy()
        df_encoded.loc[val_idx, encoded_col] = df_encoded.loc[val_idx, feature_col].map(fold_means).fillna(global_mean)

    return df_encoded

# Apply target encoding to category
print("Target encoding category...")
df_processed = target_encode_feature(df_processed, 'category', 'price', n_folds=5)

# Apply target encoding to brand
print("Target encoding brand...")
df_processed = target_encode_feature(df_processed, 'brand', 'price', n_folds=5)

# Validate target encoding
print(f"\nTarget encoding validation:")
print(f"- category_encoded range: {df_processed['category_encoded'].min():.2f} - {df_processed['category_encoded'].max():.2f}")
print(f"- brand_encoded range: {df_processed['brand_encoded'].min():.2f} - {df_processed['brand_encoded'].max():.2f}")

# Check correlation with target (should be positive but not too high to avoid overfitting)
cat_corr = df_processed['category_encoded'].corr(df_processed['price'])
brand_corr = df_processed['brand_encoded'].corr(df_processed['price'])

print(f"- category_encoded correlation with price: {cat_corr:.4f}")
print(f"- brand_encoded correlation with price: {brand_corr:.4f}")

# Distribution checks
print(f"\nCategory encoding distribution:")
print(df_processed.groupby('category')['category_encoded'].mean().sort_values(ascending=False).head())

print(f"\nBrand encoding distribution (top 5):")
brand_encoding_summary = df_processed.groupby('brand')['brand_encoded'].mean().sort_values(ascending=False)
print(brand_encoding_summary.head())

print("✅ Target encoding completed")

STEP 7: TARGET ENCODING (OUT-OF-FOLD)
Target encoding category...
Target encoding brand...

Target encoding validation:
- category_encoded range: 6.10 - 2142.00
- brand_encoded range: 6.10 - 2446.02
- category_encoded correlation with price: 0.6912
- brand_encoded correlation with price: 0.6775

Category encoding distribution:
category
laptops         2085.229811
handbags         224.985872
smartwatches     221.601862
golf             186.717699
smartphones      145.607526
Name: category_encoded, dtype: float64

Brand encoding distribution (top 5):
brand
Macbook     2375.776308
Apple        536.354652
Taylor       399.248496
Callaway     327.248960
Mizuno       313.562496
Name: brand_encoded, dtype: float64
✅ Target encoding completed


In [44]:
# ── Step 8: Feature Scaling & Final Processing ──
print("=" * 70)
print("STEP 8: FEATURE SCALING & FINAL PROCESSING")
print("=" * 70)

# Define feature types
numeric_features = [
    'condition_bucket', 'title_length', 'seller_feedback_score', 'seller_feedback_pct',
    'group_median_price', 'group_mean_price', 'group_std_price', 'group_count',
    'category_encoded', 'brand_encoded'
]

# Add title embedding features
embedding_features = [f'title_emb_{i}' for i in range(16)]
numeric_features.extend(embedding_features)

# Features to scale (exclude encoded features and counts)
features_to_scale = [
    'title_length', 'seller_feedback_score', 'seller_feedback_pct',
    'group_median_price', 'group_mean_price', 'group_std_price'
] + embedding_features

print(f"Features to scale: {len(features_to_scale)}")
print(f"Total numeric features: {len(numeric_features)}")

# Handle missing values in numeric features (from group stats)
numeric_fill_values = {
    'group_median_price': df_processed['price'].median(),
    'group_mean_price': df_processed['price'].mean(),
    'group_std_price': df_processed['price'].std(),
    'group_count': 1
}

for col, fill_val in numeric_fill_values.items():
    if col in df_processed.columns:
        missing_count = df_processed[col].isnull().sum()
        if missing_count > 0:
            df_processed[col] = df_processed[col].fillna(fill_val)
            print(f"Filled {missing_count} missing values in {col} with {fill_val:.2f}")

# Apply standard scaling
scaler = StandardScaler()
df_processed[[f + '_scaled' for f in features_to_scale]] = scaler.fit_transform(df_processed[features_to_scale])

print(f"Applied standard scaling to {len(features_to_scale)} features")

# Create log-transformed target for modeling
df_processed['price_log'] = np.log1p(df_processed['price'])
print(f"Added log-transformed target: price_log (skewness: {df_processed['price_log'].skew():.4f})")

# ── Step 9: Train/Test Split ──
print("=" * 70)
print("STEP 9: TRAIN/TEST SPLIT")
print("=" * 70)

from sklearn.model_selection import train_test_split

# Regular random split (stratified not possible due to small categories)
train_df, test_df = train_test_split(
    df_processed,
    test_size=0.2,
    random_state=42
)

print(f"Train/test split:")
print(f"- Train: {len(train_df):,} samples ({len(train_df)/len(df_processed)*100:.1f}%)")
print(f"- Test: {len(test_df):,} samples ({len(test_df)/len(df_processed)*100:.1f}%)")

# Validate stratification
print(f"\nCategory distribution check:")
train_cat_dist = train_df['category'].value_counts(normalize=True).sort_index()
test_cat_dist = test_df['category'].value_counts(normalize=True).sort_index()
dist_diff = (train_cat_dist - test_cat_dist).abs().max()
print(f"Maximum category distribution difference: {dist_diff:.4f} (should be < 0.01)")

# ── Step 10: Save Processed Data ──
print("=" * 70)
print("STEP 10: SAVE PROCESSED DATA")
print("=" * 70)

# Create output directory
output_dir = Path('../datasets/processed')
output_dir.mkdir(exist_ok=True)

# Save processed datasets
train_path = output_dir / 'ebay_train_processed.csv'
test_path = output_dir / 'ebay_test_processed.csv'
full_path = output_dir / 'ebay_full_processed.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
df_processed.to_csv(full_path, index=False)

print(f"Saved processed datasets:")
print(f"- Train: {train_path} ({len(train_df):,} rows)")
print(f"- Test: {test_path} ({len(test_df):,} rows)")
print(f"- Full: {full_path} ({len(df_processed):,} rows)")

# Save feature lists for modeling
feature_info = {
    'numeric_features': numeric_features,
    'features_to_scale': features_to_scale,
    'scaled_features': [f + '_scaled' for f in features_to_scale],
    'categorical_features': ['category', 'brand', 'condition'],
    'embedding_features': embedding_features,
    'target_features': ['price', 'price_log'],
    'metadata': {
        'total_samples': len(df_processed),
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'categories': df_processed['category'].nunique(),
        'brands': df_processed['brand'].nunique(),
        'price_range': [df_processed['price'].min(), df_processed['price'].max()],
        'processing_date': pd.Timestamp.now().isoformat()
    }
}

import json
feature_info_path = output_dir / 'feature_info.json'
with open(feature_info_path, 'w') as f:
    json.dump(feature_info, f, indent=2, default=str)

print(f"- Feature info: {feature_info_path}")

# ── Final Summary ──
print("=" * 70)
print("PREPROCESSING COMPLETED SUCCESSFULLY!")
print("=" * 70)

print(f"""
FINAL DATASET SUMMARY:
- Total samples: {len(df_processed):,}
- Features: {len(df_processed.columns)}
- Categories: {df_processed['category'].nunique()}
- Brands extracted: {df_processed['brand'].nunique()}

FEATURES CREATED:
- Condition buckets: {df_processed['condition_bucket'].nunique()} levels
- Title embeddings: 16 PCA components
- Comparable stats: median, mean, std, count per group
- Target encoding: category and brand (out-of-fold)
- Scaled features: {len(features_to_scale)} standardized

TARGET VARIABLES:
- price: original (skewness: {df_processed['price'].skew():.2f})
- price_log: log-transformed (skewness: {df_processed['price_log'].skew():.2f})

READY FOR MODELING! 🚀
""")

print(f"Next steps:")
print(f"1. Load processed data from {output_dir}")
print(f"2. Use feature_info.json for feature lists")
print(f"3. Start with price_log as target for better model performance")
print(f"4. Consider feature selection and engineering based on model results")

STEP 8: FEATURE SCALING & FINAL PROCESSING
Features to scale: 22
Total numeric features: 26
Applied standard scaling to 22 features
Added log-transformed target: price_log (skewness: 1.0148)
STEP 9: TRAIN/TEST SPLIT
Train/test split:
- Train: 5,406 samples (80.0%)
- Test: 1,352 samples (20.0%)

Category distribution check:
Maximum category distribution difference: 0.0129 (should be < 0.01)
STEP 10: SAVE PROCESSED DATA
Saved processed datasets:
- Train: ../datasets/processed/ebay_train_processed.csv (5,406 rows)
- Test: ../datasets/processed/ebay_test_processed.csv (1,352 rows)
- Full: ../datasets/processed/ebay_full_processed.csv (6,758 rows)
- Feature info: ../datasets/processed/feature_info.json
PREPROCESSING COMPLETED SUCCESSFULLY!

FINAL DATASET SUMMARY:
- Total samples: 6,758
- Features: 62
- Categories: 22
- Brands extracted: 59

FEATURES CREATED:
- Condition buckets: 5 levels
- Title embeddings: 16 PCA components
- Comparable stats: median, mean, std, count per group
- Target en